# DocuRoute — train the final Layer 2 model on Kaggle

Trains **only the final model**: one run over all 19 documents, with the
per-label thresholds taken from the five cross-validation folds rather than
tuned here. The folds themselves were trained on Colab and are not repeated.

Before running, in the panel on the right:

- **Settings → Accelerator → GPU** (T4 or P100). The notebook stops if there is
  no GPU rather than training on the CPU for hours.
- **Settings → Internet → On.** Needed to clone the repository and install
  transformers. Kaggle asks for phone verification the first time.
- **Add Input → Datasets** → attach the dataset holding the fold thresholds, so
  it appears at `/kaggle/input/docuroute-folds/`. It must contain
  `fold_0/thresholds.json` … `fold_4/thresholds.json`.

Output: `/kaggle/working/context_v2_final.zip`, which unzips straight into
`Backend/ml/saved_models/context_v2/`.

## 1. GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Settings -> Accelerator -> GPU, then Run All again.\n"
        "Training this on the CPU takes hours and is never what you want here."
    )

name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name}  ({total_gb:.1f} GB)")
print("torch:", torch.__version__)

## 2. Clone the repository

In [ ]:
import os, shutil, subprocess

REPO = "https://github.com/Doculan/DocuRoute1.git"
BRANCH = "main"
CHECKOUT = "/kaggle/working/DocuRoute1"

# Step out before deleting, so a rerun does not leave the shell without a
# working directory.
os.chdir("/kaggle/working")
if os.path.exists(CHECKOUT):
    shutil.rmtree(CHECKOUT)

result = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CHECKOUT],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)
if result.returncode != 0 or not os.path.exists(f"{CHECKOUT}/Backend"):
    raise SystemExit(
        "Clone failed - nothing below this cell will work.\n"
        "Check Settings -> Internet is On. The repository is public, so no "
        "token is needed."
    )

os.chdir(f"{CHECKOUT}/Backend")
print("working directory:", os.getcwd())

## 3. Dependencies

Kaggle already ships torch, pandas and scikit-learn. Only the Hugging Face
packages are installed, and only if they are missing - the full
`requirements.txt` carries Django and the PDF stack, which have no part in
training and take minutes to install.

In [ ]:
import importlib, subprocess, sys

needed = []
for module, package in (("transformers", "transformers"),
                        ("datasets", "datasets"),
                        ("accelerate", "accelerate"),
                        ("sklearn", "scikit-learn")):
    try:
        importlib.import_module(module)
    except ImportError:
        needed.append(package)

if needed:
    print("installing:", ", ".join(needed))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)
else:
    print("nothing to install")

import transformers, sklearn
print("transformers", transformers.__version__, "| scikit-learn", sklearn.__version__)

## 4. Fold thresholds

Taken from the attached dataset, not tuned here. The final model keeps only an
8% slice for early stopping, which is far too small to set a cut-off for a rare
label - a couple of examples either way would move it wildly. Each fold tuned
on a proper validation split, so their median is the honest estimate.

In [ ]:
import pathlib

FOLDS = pathlib.Path("/kaggle/input/docuroute-folds")
if not FOLDS.exists():
    raise SystemExit(
        "/kaggle/input/docuroute-folds not found.\n"
        "Add Input -> Datasets, attach the fold dataset, and make sure its "
        "slug gives that path."
    )

threshold_files = sorted(FOLDS.glob("fold_*/thresholds.json"))
if not threshold_files:
    threshold_files = sorted(FOLDS.glob("*/thresholds.json"))
if not threshold_files:
    raise SystemExit(
        f"No fold_*/thresholds.json under {FOLDS}. Contents: "
        f"{[p.name for p in FOLDS.iterdir()]}"
    )

print(f"{len(threshold_files)} fold threshold files:")
for path in threshold_files:
    print("  ", path.relative_to(FOLDS))

## 5. Train

The same settings the folds ran with, so the final model is that training run
over more data rather than a different one. `MAX_LENGTH` comes from
`config`, which is the only place it is defined - the Colab notebook once
pinned it separately and trained five folds at the wrong length.

In [ ]:
import json, pathlib, random, subprocess, sys, time

sys.path.insert(0, "ml")
from revision_pipeline import config

EPOCHS = 3
BATCH = 16
PATIENCE = 2
MAX_LENGTH = config.MAX_LENGTH
print(f"epochs={EPOCHS} batch={BATCH} patience={PATIENCE} max_length={MAX_LENGTH}")

FINAL_DIR = pathlib.Path("ml/saved_models/context_v2/final")
FINAL_DATA = pathlib.Path("ml/datasets/context_v2/_final")
FINAL_DATA.mkdir(parents=True, exist_ok=True)

rows = [json.loads(line) for line in
        pathlib.Path("ml/datasets/context_v2/all.jsonl").open(encoding="utf-8")
        if line.strip()]
random.Random(42).shuffle(rows)
cut = max(int(len(rows) * 0.08), 1)
(FINAL_DATA / "val.jsonl").write_text(
    "\n".join(json.dumps(r) for r in rows[:cut]), encoding="utf-8")
(FINAL_DATA / "train.jsonl").write_text(
    "\n".join(json.dumps(r) for r in rows[cut:]), encoding="utf-8")
print(f"{len(rows) - cut} train / {cut} held out for early stopping")

# -u so the per-epoch lines appear while it trains rather than all at the end.
started = time.time()
subprocess.run([
    sys.executable, "-u", "ml/revision_pipeline/scripts/train_layer2.py",
    "--data-dir", str(FINAL_DATA), "--out-dir", str(FINAL_DIR),
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH),
    "--max-length", str(MAX_LENGTH), "--device", "cuda",
    "--patience", str(PATIENCE),
    "--thresholds-from", str(FOLDS),
], check=True)
print(f"finished in {(time.time() - started) / 60:.1f} min")

## 6. Package the model

The archive's root is the **contents** of the model directory, so it unzips
straight into `Backend/ml/saved_models/context_v2/` with nothing to move
afterwards.

In [ ]:
import pathlib, zipfile

ARCHIVE = pathlib.Path("/kaggle/working/context_v2_final.zip")
if ARCHIVE.exists():
    ARCHIVE.unlink()

expected = ["encoder", "tokenizer", "heads.pt", "label_config.json"]
missing = [name for name in expected if not (FINAL_DIR / name).exists()]
if missing:
    raise SystemExit(f"training did not produce: {missing}")

with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(FINAL_DIR.rglob("*")):
        if path.is_file():
            # Relative to FINAL_DIR, so "encoder/..." sits at the archive root.
            archive.write(path, path.relative_to(FINAL_DIR))

print(f"{ARCHIVE}  ({ARCHIVE.stat().st_size / 1e6:.0f} MB)")
print("\ntop-level entries:")
with zipfile.ZipFile(ARCHIVE) as archive:
    print(sorted({name.split("/")[0] for name in archive.namelist()}))
print("\nDownload it from the Output panel on the right, then unzip into")
print("Backend/ml/saved_models/context_v2/")